In [1]:
import fitz
import io
import torch
import transformers
from PIL import Image
from huggingface_hub.utils import enable_progress_bars
from colpali_engine.models import ColQwen2, ColQwen2Processor
import psycopg2
from pgvector.psycopg2 import register_vector

from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption

c:\Users\UserAdmin\Documents\RAG\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
transformers.utils.logging.set_verbosity_info()
enable_progress_bars()

In [3]:
pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.mode = "accurate"

In [4]:
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

In [5]:
result = converter.convert("pdfs/government-data-security-policies.pdf")
doc = result.document

[INFO] 2026-04-14 17:21:46,987 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-14 17:21:46,995 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-04-14 17:21:46,995 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\UserAdmin\Documents\RAG\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-14 17:21:46,995 [RapidOCR] main.py:50: Using C:\Users\UserAdmin\Documents\RAG\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-14 17:21:47,193 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-14 17:21:47,193 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-04-14 17:21:47,193 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\UserAdmin\Documents\RAG\venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-04-14 17:21:47,193 [RapidOCR] main.py:50: Using C:\Users\UserAdmin\Documents\RAG\venv\Lib\site-packages\

In [6]:
md_index = []

In [7]:
for page_number, page in doc.pages.items():
    page_md = doc.export_to_markdown(page_no=page_number)
    md_index.append({
        "page_number": page_number,
        "page_md": page_md
    }) 

In [8]:
fitz_doc = fitz.open("pdfs/government-data-security-policies.pdf")

In [9]:
model_name = "vidore/colqwen2-v1.0"
model = ColQwen2.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
).eval()


loading configuration file config.json from cache at C:\Users\UserAdmin\.cache\huggingface\hub\models--vidore--colqwen2-base\snapshots\9fe8a713422a7cb4ef79ca77a09b381ee2243101\config.json
Model config Qwen2VLConfig {
  "architectures": [
    "ColQwen2"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "float32",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "image_token_id": 151655,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "max_position_embeddings": 32768,
  "max_window_layers": 28,
  "model_type": "qwen2_vl",
  "num_attention_heads": 12,
  "num_hidden_layers": 28,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "rope_scaling": {
    "mrope_section": [
      16,
      24,
      24
    ],
    "rope_type": "default",
    "type": "default"
  },
  "rope_theta": 1000000.0,
  "sliding_window": 32768,
  "text_config": {
    "_name_or_path": "/lus/home/CT10/cad15443/mfaysse/colpali/models/Qwen2-VL-2B-Instruct",
    "arc

In [10]:
processor = ColQwen2Processor.from_pretrained(
    model_name, 
    trust_remote_code=True
)

loading configuration file preprocessor_config.json from cache at C:\Users\UserAdmin\.cache\huggingface\hub\models--vidore--colqwen2-v1.0\snapshots\83a0134c8f274b3688d8dbde26de8a5b109ad8b4\preprocessor_config.json
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
Image processor Qwen2VLImageProcessorFast {
  "crop_size": null,
  "data_format": "channels_first",
  "default_to_square": true,
  "device": null,
  "disable_grouping": null,
  "do_center_crop": null,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_pad": null,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.48145466,
    0.4578275,
    0.40821073
  ],
  "image_proces

In [11]:
vector_index = []

In [12]:
for i in range(len(fitz_doc)):
    page = fitz_doc.load_page(i)
    pix = page.get_pixmap(dpi=150)
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    
    inputs = processor.process_images(images=[img])
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        embeddings = model(**inputs)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)

        coarse_vector = embeddings.mean(dim=1).squeeze()
    
    vector_index.append({
        "page_number": i+1,
        "coarse_vector": coarse_vector.cpu(),
        "page_embeddings": embeddings.cpu()
        
    })


docker exec -it pgvector-db psql -U postgres

In [13]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="rag_db",
    user="postgres"
)
register_vector(conn)

In [15]:
def sanitize_string(s: str) -> str:
    if s is None:
        return None
    return s.replace('\x00', '')

In [22]:
vector_index[0]["coarse_vector"].detach().cpu().flatten().tolist()

[0.09326171875,
 -0.00946044921875,
 0.061279296875,
 0.01318359375,
 0.0245361328125,
 -0.028076171875,
 -0.03515625,
 -0.00543212890625,
 0.01129150390625,
 0.0859375,
 -0.05712890625,
 0.01214599609375,
 -0.0062255859375,
 -0.00823974609375,
 0.04736328125,
 0.021728515625,
 0.06201171875,
 0.0078125,
 0.0029144287109375,
 0.026611328125,
 -0.036376953125,
 0.0308837890625,
 0.004852294921875,
 -0.072265625,
 -0.03857421875,
 -0.028564453125,
 -0.018310546875,
 -0.025146484375,
 -0.07080078125,
 0.013916015625,
 0.034912109375,
 0.02978515625,
 -0.04150390625,
 0.012451171875,
 -0.036376953125,
 -0.002716064453125,
 -0.03173828125,
 0.061279296875,
 -0.036376953125,
 0.00555419921875,
 0.047119140625,
 -0.05908203125,
 0.0205078125,
 0.006439208984375,
 -0.09326171875,
 -0.004852294921875,
 0.006927490234375,
 -0.018310546875,
 -0.0634765625,
 0.044677734375,
 0.0142822265625,
 -0.026611328125,
 -0.09716796875,
 0.040283203125,
 0.032470703125,
 -0.0517578125,
 0.051513671875,
 -0.0

In [20]:
try:     
    with conn.cursor() as cur:
        for i, item in enumerate(vector_index):
            page_md = sanitize_string(md_index[i]["page_md"])

            cur.execute(
                '''
                INSERT INTO pages(page_number, coarse_vector, markdown)
                VALUES (%s, %s, %s) RETURNING id;
                ''',
                (item["page_number"], item["coarse_vector"].detach().cpu().flatten().tolist(), page_md)
            )

            page_id = cur.fetchone()[0]

            patch_vectors = item["page_embeddings"].tolist()

            patch_rows = []
            for index, vec in enumerate(patch_vectors):
                patch_rows.append((
                    page_id, 
                    index, 
                    vec
                ))

            cur.executemany(
                '''
                INSERT INTO page_patches (page_id, page_index, patch_embedding)
                VALUES (%s, %s, %s);
                ''',
                patch_rows
            )

        conn.commit()

except Exception as e: 
        print(f"Error: {e}")
        conn.rollback()

Error: array must be 1-D

